# RHI LoRA v3 Training Builder
## `slot_builder_lora_v3` + `shape_critic_v1`

This notebook backs up to the confirmed v8 corpus and builds the real v9 path:

```text
v8 PATCHED corpus
  ↓
repair rows → slot_builder_lora_v3 dataset
shape rows  → shape_critic_v1 dataset
  ↓
train adapters
  ↓
v9 shape-guided synthesis runtime
```

Core lock:

$$
\boxed{\text{Prediction beats enumeration. Synthesis beats consensus.}}
$$

v9 target:

$$
Q
\rightarrow
C_{\text{v3}}
\rightarrow
K^*_{\text{shape critic}}
\rightarrow
A_{\text{shape-guided}}
\rightarrow
\Psi/\Omega.
$$


In [ ]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path

ROOT = Path.cwd()

CANDIDATE_INPUT_DIRS = [
    ROOT,
    ROOT / "rhi_live_runtime_v8_outputs",
    ROOT / "rhi_live_runtime_v8_5_outputs",
    ROOT / "rhi_live_runtime_v7_outputs",
]

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
LOCAL_FILES_ONLY = True

DATASET_DIR = ROOT / "rhi_lora_v3_training_data"
SLOT_OUTPUT_DIR = ROOT / "slot_builder_lora_v3"
SHAPE_OUTPUT_DIR = ROOT / "shape_critic_v1"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

BUILD_DATASETS = True
RUN_TRAINING = False          # Set True only when ready to train.
INSTALL_MISSING = False       # Set True if imports fail.
USE_4BIT = False

MAX_LENGTH_SLOT = 2048
MAX_LENGTH_SHAPE = 2048

SLOT_LORA_RANK = 16
SLOT_LORA_ALPHA = 32
SLOT_LORA_DROPOUT = 0.05
SLOT_TARGET_MODULES = ["q_proj", "v_proj"]
SLOT_EPOCHS = 5
SLOT_LR = 3e-4

SHAPE_LORA_RANK = 8
SHAPE_LORA_ALPHA = 16
SHAPE_LORA_DROPOUT = 0.05
SHAPE_TARGET_MODULES = ["q_proj", "k_proj", "v_proj"]
SHAPE_EPOCHS = 8
SHAPE_LR = 5e-4

print("ROOT:", ROOT)
print("DATASET_DIR:", DATASET_DIR)
print("RUN_TRAINING:", RUN_TRAINING)


In [ ]:
# ============================================================
# IMPORTS
# ============================================================
if INSTALL_MISSING:
    import sys, subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-U",
        "transformers", "peft", "accelerate", "sentencepiece", "datasets", "pandas"
    ])

import json, re, os, math, time
from typing import Any, Dict, List, Tuple, Optional
from collections import defaultdict, Counter
import pandas as pd

print("pandas:", pd.__version__)


In [ ]:
# ============================================================
# FILE DISCOVERY
# ============================================================
def first_existing(names: List[str], dirs: List[Path]) -> Optional[Path]:
    for d in dirs:
        for name in names:
            p = d / name
            if p.exists():
                return p
    return None

REPAIR_FILE = first_existing([
    "rhi_repair_training_rows_v8.jsonl",
    "rhi_repair_training_rows_v8_5.jsonl",
    "rhi_repair_training_rows_v7.jsonl",
], CANDIDATE_INPUT_DIRS)

SHAPE_FILE = first_existing([
    "rhi_shape_training_rows_v8.jsonl",
    "rhi_shape_training_rows_v8_5.jsonl",
    "rhi_shape_training_rows_v7.jsonl",
], CANDIDATE_INPUT_DIRS)

RUNS_FILE = first_existing([
    "rhi_live_runs_v8.jsonl",
    "rhi_live_runs_v8_5.jsonl",
    "rhi_live_runs_v7.jsonl",
], CANDIDATE_INPUT_DIRS)

print("REPAIR_FILE:", REPAIR_FILE)
print("SHAPE_FILE :", SHAPE_FILE)
print("RUNS_FILE  :", RUNS_FILE)

if REPAIR_FILE is None:
    print("WARNING: repair rows not found. Put rhi_repair_training_rows_v8.jsonl beside this notebook or inside rhi_live_runtime_v8_outputs/.")
if SHAPE_FILE is None:
    print("WARNING: shape rows not found. Put rhi_shape_training_rows_v8.jsonl beside this notebook or inside rhi_live_runtime_v8_outputs/.")
if RUNS_FILE is None:
    print("NOTE: live runs file not found. Full contracts are better when rhi_live_runs_v8.jsonl is available.")


In [ ]:
# ============================================================
# JSONL HELPERS + PROMPTS
# ============================================================
REQUIRED_FIELDS = [
    "family_class", "domain_carrier", "forbidden_neighbor_carrier",
    "boundary_conditions", "preserved_function", "failure_modes",
    "witness_readout", "residue",
]

SLOT_SYSTEM_PROMPT = (
    "You are the Nexus Slot Constructor. Generate missing-shape contracts. "
    "Required fields: family_class, domain_carrier, forbidden_neighbor_carrier, "
    "boundary_conditions, preserved_function, failure_modes, witness_readout, residue. "
    "family_class must be a complete noun phrase. Use operational fit, not labels."
)

SHAPE_CRITIC_SYSTEM_PROMPT = (
    "You are the Nexus Shape Critic. Given a contract and prompt, predict the dominant "
    "operation-shape and detect composites. Output strict JSON with: dominant_shape, "
    "dominant_mass, composite, shape_field_mass, normalized_masses, confidence."
)

SHAPE_ONTOLOGY = {
    "FILTER": {"verbs": ["remove", "exclude", "screen", "reject", "discard", "block"], "nouns": ["invalid", "noise", "candidate", "state", "set", "selection"], "gloss": "excludes or removes invalid states from a candidate set"},
    "GATE": {"verbs": ["allow", "permit", "open", "close", "block", "transition", "cross"], "nouns": ["condition", "permission", "transition", "threshold", "entry", "passage", "interface"], "gloss": "conditionally permits or blocks transition across a boundary"},
    "LOCK": {"verbs": ["prevent", "hold", "require", "unlock", "release"], "nouns": ["key", "condition", "constraint", "permission", "state"], "gloss": "prevents transition until a key or condition is satisfied"},
    "CONTRACT": {"verbs": ["bind", "commit", "constrain", "specify"], "nouns": ["intent", "terms", "boundary", "obligation", "precondition", "action"], "gloss": "binds future action to prior conditions and intent"},
    "BOUNDARY": {"verbs": ["separate", "define", "cross", "limit", "constrain"], "nouns": ["interface", "edge", "condition", "crossing", "limit", "domain"], "gloss": "defines the valid interface crossing between states"},
    "GROOVE": {"verbs": ["lower", "bias", "guide", "adapt"], "nouns": ["path", "resistance", "adapter", "rank", "manifold", "delta", "lora"], "gloss": "lowers resistance along a preferred path without rewriting the whole field"},
    "RESIDUE": {"verbs": ["remain", "repair", "backpatch", "capture"], "nouns": ["mismatch", "failure", "trace", "error", "signal", "memory"], "gloss": "unresolved mismatch left after collapse, captured for repair"},
}

def read_jsonl(path: Optional[Path]) -> List[Dict]:
    rows = []
    if path is None or not Path(path).exists():
        return rows
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(path: Path, rows: List[Dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def normalize_list(x):
    if x is None: return []
    if isinstance(x, list): return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        s = x.strip()
        if not s: return []
        return [p.strip() for p in re.split(r"[|,;]", s) if p.strip()]
    return [str(x).strip()]

def stable_dedupe(items):
    out, seen = [], set()
    for x in items:
        s = str(x).strip()
        if s and s not in seen:
            seen.add(s); out.append(s)
    return out

def normalize_contract(c0: Optional[Dict]) -> Dict:
    c0 = c0 or {}
    return {
        "family_class": str(c0.get("family_class", "") or "").strip(),
        "domain_carrier": normalize_list(c0.get("domain_carrier", [])),
        "forbidden_neighbor_carrier": normalize_list(c0.get("forbidden_neighbor_carrier", [])),
        "boundary_conditions": normalize_list(c0.get("boundary_conditions", [])),
        "preserved_function": str(c0.get("preserved_function", "") or "").strip(),
        "failure_modes": normalize_list(c0.get("failure_modes", [])),
        "witness_readout": str(c0.get("witness_readout", "") or "").strip(),
        "residue": c0.get("residue", None),
    }

def contract_complete(c: Dict) -> bool:
    c = normalize_contract(c)
    for f in REQUIRED_FIELDS:
        if f == "residue": continue
        v = c.get(f)
        if isinstance(v, list) and len(v) == 0: return False
        if not isinstance(v, list) and not str(v or "").strip(): return False
    return True

def build_slot_user_prompt(prompt: str) -> str:
    return "\n".join([
        "Prompt: " + str(prompt), "",
        "Generate the missing-shape contract.",
        "Checklist:",
        "1. Need: occupy the inverse cavity.",
        "2. Function: preserve or redirect the required operation.",
        "3. Boundary: respect constraints.",
        "4. Trap: reject noun/surface-label confusion.",
        "5. Collapse: produce one executable witness/readout.",
        "family_class must be a complete noun phrase, not a dangling preposition.",
        "Return JSON only.",
    ])

def apply_chat_template_fallback(messages: List[Dict[str, str]]) -> str:
    return "\n\n".join([m["role"].upper() + ":\n" + m["content"] for m in messages])


In [ ]:
# ============================================================
# LOAD AND INSPECT v8 CORPUS
# ============================================================
repair_rows = read_jsonl(REPAIR_FILE)
shape_rows  = read_jsonl(SHAPE_FILE)
run_rows    = read_jsonl(RUNS_FILE)

print("repair rows:", len(repair_rows))
print("shape rows :", len(shape_rows))
print("run rows   :", len(run_rows))

if repair_rows:
    repair_df = pd.DataFrame(repair_rows)
    print("\nRepair type counts:")
    display(repair_df.groupby("repair_type").size().reset_index(name="count").sort_values("count", ascending=False))
else:
    repair_df = pd.DataFrame()

if shape_rows:
    shape_df = pd.DataFrame(shape_rows)
    group_cols = [c for c in ["best_shape", "runtime_reason", "composite_detected"] if c in shape_df.columns]
    if group_cols:
        print("\nShape row counts:")
        display(shape_df.groupby(group_cols).size().reset_index(name="count").sort_values("count", ascending=False))
else:
    shape_df = pd.DataFrame()

if run_rows:
    run_summary = []
    for r in run_rows:
        res = r.get("resolution", {})
        run_summary.append({
            "run_id": r.get("run_id"),
            "state": res.get("state"),
            "reason": res.get("reason"),
            "prompt": str(r.get("prompt", ""))[:80],
            "contract_complete": (r.get("contract_result") or {}).get("complete"),
        })
    display(pd.DataFrame(run_summary))


In [ ]:
# ============================================================
# RECONSTRUCT CONTRACTS FROM LIVE RUNS OR REPAIR ROWS
# ============================================================
def contract_from_run(row: Dict) -> Optional[Dict]:
    cr = row.get("contract_result") or {}
    c = cr.get("contract")
    return normalize_contract(c) if isinstance(c, dict) else None

def build_contract_index_from_runs(run_rows: List[Dict]) -> Dict[str, Dict]:
    out = {}
    for r in run_rows:
        run_id = r.get("run_id")
        c = contract_from_run(r)
        if run_id and c:
            out[run_id] = c
    return out

def reconstruct_contract_from_repairs(repairs: List[Dict]) -> Dict:
    c = {"family_class":"", "domain_carrier":[], "forbidden_neighbor_carrier":[], "boundary_conditions":[], "preserved_function":"", "failure_modes":[], "witness_readout":"", "residue":None}
    for row in repairs:
        field = row.get("field")
        rtype = row.get("repair_type")
        good = row.get("good_value")
        if field in {"domain_carrier", "forbidden_neighbor_carrier", "boundary_conditions", "failure_modes"}:
            c[field] = stable_dedupe(c.get(field, []) + normalize_list(good))
        elif field in c and good is not None:
            c[field] = " ".join(map(str, good)) if isinstance(good, list) else str(good)
        if rtype == "positive_injection":
            c["domain_carrier"] = stable_dedupe(c["domain_carrier"] + normalize_list(good))
        elif rtype == "forbidden_injection":
            c["forbidden_neighbor_carrier"] = stable_dedupe(c["forbidden_neighbor_carrier"] + normalize_list(good))
        elif rtype == "polarity_rewrite":
            c["boundary_conditions"] = normalize_list(good)
    prompt = repairs[0].get("prompt", "") if repairs else ""
    words = [w for w in re.findall(r"[A-Za-z0-9_]+", prompt.lower()) if len(w) > 3]
    if not c["family_class"]: c["family_class"] = "operational closure"
    if not c["domain_carrier"]: c["domain_carrier"] = stable_dedupe(words[:8])
    if not c["forbidden_neighbor_carrier"]: c["forbidden_neighbor_carrier"] = ["surface label without operational fit"]
    if not c["boundary_conditions"]: c["boundary_conditions"] = ["answer must preserve the requested operation and reject surface-label confusion"]
    if not c["preserved_function"]: c["preserved_function"] = "preserve the prompt's requested operation"
    if not c["failure_modes"]: c["failure_modes"] = ["generic answer", "label-only answer"]
    if not c["witness_readout"]: c["witness_readout"] = "answer demonstrates the operation using domain-specific verbs and nouns"
    return normalize_contract(c)

contract_by_run = build_contract_index_from_runs(run_rows)
repairs_by_run = defaultdict(list)
for row in repair_rows:
    repairs_by_run[row.get("run_id", "unknown")].append(row)

print("contracts from runs:", len(contract_by_run))
print("repair run groups:", len(repairs_by_run))

preview = []
for run_id, repairs in list(repairs_by_run.items())[:10]:
    c = contract_by_run.get(run_id) or reconstruct_contract_from_repairs(repairs)
    preview.append({"run_id":run_id, "source":"live_run" if run_id in contract_by_run else "reconstructed", "complete":contract_complete(c), "family_class":c.get("family_class"), "domain_n":len(c.get("domain_carrier", [])), "prompt":repairs[0].get("prompt", "")[:70]})
display(pd.DataFrame(preview))


In [ ]:
# ============================================================
# BUILD slot_builder_lora_v3 DATASET
# ============================================================
def build_slot_training_examples(repair_rows: List[Dict], run_rows: List[Dict]) -> List[Dict]:
    contract_by_run = build_contract_index_from_runs(run_rows)
    by_run = defaultdict(list)
    for row in repair_rows:
        by_run[row.get("run_id", "unknown")].append(row)
    examples = []
    for run_id, repairs in by_run.items():
        if not repairs: continue
        prompt = repairs[0].get("prompt", "")
        contract = contract_by_run.get(run_id) or reconstruct_contract_from_repairs(repairs)
        contract = normalize_contract(contract)
        messages = [
            {"role":"system", "content":SLOT_SYSTEM_PROMPT},
            {"role":"user", "content":build_slot_user_prompt(prompt)},
            {"role":"assistant", "content":json.dumps(contract, ensure_ascii=False)},
        ]
        examples.append({"run_id":run_id, "prompt":prompt, "target_contract":contract, "contract_source":"live_run" if run_id in contract_by_run else "reconstructed", "messages":messages})
    return examples

slot_examples = build_slot_training_examples(repair_rows, run_rows)
slot_train_path = DATASET_DIR / "slot_builder_lora_v3_train.jsonl"
write_jsonl(slot_train_path, slot_examples)

print("slot examples:", len(slot_examples))
print("saved:", slot_train_path)
if slot_examples:
    display(pd.DataFrame([{ "run_id":ex["run_id"], "source":ex["contract_source"], "complete":contract_complete(ex["target_contract"]), "family_class":ex["target_contract"]["family_class"], "prompt":ex["prompt"][:80]} for ex in slot_examples]))
    print(slot_examples[0]["messages"][-1]["content"][:1200])


In [ ]:
# ============================================================
# BUILD shape_critic_v1 DATASET
# ============================================================
def choose_shape_target(first_row: Dict) -> Dict:
    composite = first_row.get("composite")
    dominant_shape = first_row.get("dominant_shape")
    target_shape = composite.get("composite") if isinstance(composite, dict) and composite.get("composite") else dominant_shape
    masses = first_row.get("shape_field_mass") or {}
    norm = first_row.get("shape_field_normalized_mass") or first_row.get("normalized_masses") or {}
    vals = sorted([float(v) for v in norm.values()], reverse=True) if isinstance(norm, dict) else []
    confidence = (vals[0]-vals[1]) if len(vals) >= 2 else (vals[0] if vals else None)
    return {"dominant_shape":target_shape, "raw_dominant_shape":dominant_shape, "dominant_mass":first_row.get("dominant_mass"), "dominant_normalized_mass":first_row.get("dominant_normalized_mass"), "shape_field_mass":masses, "normalized_masses":norm, "composite":composite, "confidence":confidence}

def build_shape_training_examples(shape_rows: List[Dict], run_rows: List[Dict]) -> List[Dict]:
    contract_by_run = build_contract_index_from_runs(run_rows)
    by_run = defaultdict(list)
    for row in shape_rows:
        by_run[row.get("run_id", "unknown")].append(row)
    examples = []
    for run_id, rows in by_run.items():
        if not rows: continue
        first = rows[0]
        prompt = first.get("prompt", "")
        contract = first.get("contract") if isinstance(first.get("contract"), dict) else contract_by_run.get(run_id, {})
        contract = normalize_contract(contract)
        target = choose_shape_target(first)
        messages = [
            {"role":"system", "content":SHAPE_CRITIC_SYSTEM_PROMPT},
            {"role":"user", "content":"Contract:\n" + json.dumps(contract, ensure_ascii=False, indent=2) + "\n\nPrompt: " + prompt + "\n\nPredict the dominant operation-shape. Return strict JSON."},
            {"role":"assistant", "content":json.dumps(target, ensure_ascii=False)},
        ]
        examples.append({"run_id":run_id, "prompt":prompt, "contract":contract, "target":target, "messages":messages, "n_branch_rows":len(rows)})
    return examples

shape_examples = build_shape_training_examples(shape_rows, run_rows)
shape_train_path = DATASET_DIR / "shape_critic_v1_train.jsonl"
write_jsonl(shape_train_path, shape_examples)

print("shape examples:", len(shape_examples))
print("saved:", shape_train_path)
if shape_examples:
    display(pd.DataFrame([{ "run_id":ex["run_id"], "target":ex["target"]["dominant_shape"], "composite":bool(ex["target"].get("composite")), "n_branch_rows":ex["n_branch_rows"], "prompt":ex["prompt"][:80]} for ex in shape_examples]))
    print(shape_examples[0]["messages"][-1]["content"][:1200])


In [ ]:
# ============================================================
# DATASET TEXT FORMATTING
# ============================================================
def load_tokenizer_for_formatting():
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, local_files_only=LOCAL_FILES_ONLY)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def render_messages(messages: List[Dict[str, str]], tokenizer=None) -> str:
    if tokenizer is not None and hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return apply_chat_template_fallback(messages)

def write_text_dataset(examples: List[Dict], out_path: Path, tokenizer=None):
    rows = [{"run_id":ex.get("run_id"), "prompt":ex.get("prompt"), "text":render_messages(ex["messages"], tokenizer=tokenizer)} for ex in examples]
    write_jsonl(out_path, rows)
    return rows

try:
    tok_preview = load_tokenizer_for_formatting()
    print("loaded tokenizer for chat-template formatting")
except Exception as e:
    tok_preview = None
    print("tokenizer not loaded; using fallback formatting:", e)

slot_text_path = DATASET_DIR / "slot_builder_lora_v3_text.jsonl"
shape_text_path = DATASET_DIR / "shape_critic_v1_text.jsonl"
slot_text_rows = write_text_dataset(slot_examples, slot_text_path, tokenizer=tok_preview)
shape_text_rows = write_text_dataset(shape_examples, shape_text_path, tokenizer=tok_preview)

print("saved:", slot_text_path, len(slot_text_rows))
print("saved:", shape_text_path, len(shape_text_rows))
if slot_text_rows:
    print("\nSLOT TEXT PREVIEW:\n", slot_text_rows[0]["text"][:1500])
if shape_text_rows:
    print("\nSHAPE TEXT PREVIEW:\n", shape_text_rows[0]["text"][:1500])


In [ ]:
# ============================================================
# TRAINING UTILITY
# ============================================================
def train_lora_from_text_jsonl(dataset_path: Path, output_dir: Path, lora_rank: int, lora_alpha: int, lora_dropout: float, target_modules: List[str], num_epochs: int, learning_rate: float, max_length: int):
    import torch
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
    from peft import LoraConfig, get_peft_model

    rows = read_jsonl(dataset_path)
    if not rows:
        raise ValueError(f"No rows in {dataset_path}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, local_files_only=LOCAL_FILES_ONLY)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {"local_files_only": LOCAL_FILES_ONLY, "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32}
    if USE_4BIT:
        from transformers import BitsAndBytesConfig
        model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        model_kwargs["device_map"] = "auto"

    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
    if not USE_4BIT and torch.cuda.is_available():
        model = model.to("cuda")

    lora_config = LoraConfig(r=lora_rank, lora_alpha=lora_alpha, target_modules=target_modules, lora_dropout=lora_dropout, bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    ds = Dataset.from_list(rows)
    def tokenize(batch):
        toks = tokenizer(batch["text"], padding="max_length", truncation=True, max_length=max_length)
        toks["labels"] = toks["input_ids"].copy()
        return toks
    tokenized = ds.map(tokenize, batched=True, remove_columns=ds.column_names)

    args = TrainingArguments(output_dir=str(output_dir), num_train_epochs=num_epochs, per_device_train_batch_size=1, gradient_accumulation_steps=4, learning_rate=learning_rate, fp16=bool(torch.cuda.is_available()), logging_steps=1, save_strategy="epoch", optim="adamw_torch", warmup_ratio=0.1, report_to=[])
    trainer = Trainer(model=model, args=args, train_dataset=tokenized)
    trainer.train()
    model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    return {"output_dir":str(output_dir), "n_rows":len(rows), "epochs":num_epochs, "learning_rate":learning_rate, "lora_rank":lora_rank, "target_modules":target_modules}

print("training utility ready")
print("Set RUN_TRAINING=True in CONFIG to train adapters.")


In [ ]:
# ============================================================
# TRAIN slot_builder_lora_v3
# ============================================================
if RUN_TRAINING:
    if len(slot_text_rows) == 0:
        raise ValueError("No slot training rows available.")
    t0 = time.time()
    slot_train_result = train_lora_from_text_jsonl(slot_text_path, SLOT_OUTPUT_DIR, SLOT_LORA_RANK, SLOT_LORA_ALPHA, SLOT_LORA_DROPOUT, SLOT_TARGET_MODULES, SLOT_EPOCHS, SLOT_LR, MAX_LENGTH_SLOT)
    print("slot_builder_lora_v3 trained in sec:", round(time.time() - t0, 2))
    print(json.dumps(slot_train_result, indent=2))
else:
    print("RUN_TRAINING=False — skipped slot_builder_lora_v3 training.")
    print("Dataset ready:", slot_text_path)


In [ ]:
# ============================================================
# TRAIN shape_critic_v1
# ============================================================
if RUN_TRAINING:
    if len(shape_text_rows) == 0:
        raise ValueError("No shape training rows available.")
    t0 = time.time()
    shape_train_result = train_lora_from_text_jsonl(shape_text_path, SHAPE_OUTPUT_DIR, SHAPE_LORA_RANK, SHAPE_LORA_ALPHA, SHAPE_LORA_DROPOUT, SHAPE_TARGET_MODULES, SHAPE_EPOCHS, SHAPE_LR, MAX_LENGTH_SHAPE)
    print("shape_critic_v1 trained in sec:", round(time.time() - t0, 2))
    print(json.dumps(shape_train_result, indent=2))
else:
    print("RUN_TRAINING=False — skipped shape_critic_v1 training.")
    print("Dataset ready:", shape_text_path)


In [ ]:
# ============================================================
# VALIDATION + SHAPE-GUIDED PROMPT HELPER
# ============================================================
def extract_first_json_object(text: str) -> Tuple[Optional[Dict], Optional[str]]:
    text = str(text).strip()
    try:
        obj = json.loads(text)
        return (obj, None) if isinstance(obj, dict) else (None, "json_not_dict")
    except Exception:
        pass
    cleaned = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        obj = json.loads(cleaned)
        return (obj, None) if isinstance(obj, dict) else (None, "fenced_json_not_dict")
    except Exception:
        pass
    start, end = text.find("{"), text.rfind("}")
    if start >= 0 and end > start:
        try:
            obj = json.loads(text[start:end+1])
            return (obj, None) if isinstance(obj, dict) else (None, "scanned_json_not_dict")
        except Exception as e:
            return None, "json_parse_error: " + str(e)
    return None, "no_json_object_found"

def simple_contract_quality(contract: Dict) -> Dict:
    c = normalize_contract(contract)
    generic_terms = {"current", "using", "properly", "answer", "task", "prompt", "question", "response"}
    scar_terms = {"general purpose", "surface label without operational fit", "unrelated_to_prompt"}
    domain = [x.lower() for x in c["domain_carrier"]]
    forbidden = [x.lower() for x in c["forbidden_neighbor_carrier"]]
    fw = c["family_class"].split()
    family_fragment = (not c["family_class"]) or (fw[-1].lower() in {"of","for","with","to","in","by","and","or","a","an","the"} if fw else True)
    return {"complete":contract_complete(c), "generic_domain_terms":[x for x in domain if x in generic_terms], "scar_forbidden_terms":[x for x in forbidden if x in scar_terms], "family_fragment":family_fragment, "domain_n":len(c["domain_carrier"]), "forbidden_n":len(c["forbidden_neighbor_carrier"]), "boundary_n":len(c["boundary_conditions"])}

def shape_guided_answer_prompt(prompt: str, contract: Dict, target_shape: str) -> str:
    spec = SHAPE_ONTOLOGY.get(target_shape, {})
    verbs = ", ".join(spec.get("verbs", []))
    nouns = ", ".join(spec.get("nouns", []))
    gloss = spec.get("gloss", "")
    return f"""Contract:
{json.dumps(normalize_contract(contract), ensure_ascii=False, indent=2)}

Target operation-shape: {target_shape}
Gloss: {gloss}
Key verbs: {verbs}
Key nouns: {nouns}

User prompt:
{prompt}

Answer through the lens of {target_shape}, using target verbs and nouns.
Do not merely label it as \"{target_shape}\" — demonstrate the operation."""

print("validation helpers ready")
if slot_examples:
    print("first slot target quality:")
    print(json.dumps(simple_contract_quality(slot_examples[0]["target_contract"]), indent=2))
if shape_examples:
    print("first shape target:")
    print(json.dumps(shape_examples[0]["target"], indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# MANIFEST
# ============================================================
manifest = {
    "notebook": "rhi_lora_v3_training_builder",
    "root": str(ROOT),
    "model_name": MODEL_NAME,
    "input_files": {"repair_file": str(REPAIR_FILE) if REPAIR_FILE else None, "shape_file": str(SHAPE_FILE) if SHAPE_FILE else None, "runs_file": str(RUNS_FILE) if RUNS_FILE else None},
    "dataset_files": {"slot_examples": str(slot_train_path), "shape_examples": str(shape_train_path), "slot_text": str(slot_text_path), "shape_text": str(shape_text_path)},
    "counts": {"repair_rows": len(repair_rows), "shape_rows": len(shape_rows), "run_rows": len(run_rows), "slot_examples": len(slot_examples), "shape_examples": len(shape_examples)},
    "adapter_outputs": {"slot_builder_lora_v3": str(SLOT_OUTPUT_DIR), "shape_critic_v1": str(SHAPE_OUTPUT_DIR)},
    "training_enabled": RUN_TRAINING,
    "next_runtime": "Q -> C_v3 -> K*_shape_critic -> A_shape_guided -> audit -> Ψ/Ω",
}
manifest_path = DATASET_DIR / "rhi_lora_v3_training_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print("saved manifest:", manifest_path)


# Ψ / Ω

If `RUN_TRAINING=False`, this notebook still completes the first fold:

```text
repair rows → slot_builder_lora_v3 dataset
shape rows  → shape_critic_v1 dataset
```

When ready:

```text
set RUN_TRAINING=True
restart kernel
run top-to-bottom
```

Outputs:

```text
slot_builder_lora_v3/
shape_critic_v1/
rhi_lora_v3_training_data/
```

Next notebook after adapters exist:

```text
rhi_v9_shape_guided_runtime.ipynb
```
